<a href="https://colab.research.google.com/github/goitstudent123/DL4CV-NLP/blob/main/final_HAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
os.environ["KAGGLE_API_TOKEN"] = "KGAT_b9a82ee271040b584349afe803c42d6f"
os.environ["KAGGLE_USERNAME"] = "antoniohusiev"

In [15]:
# Stage 1: Setup and access check
import os

!pip -q install -U "kaggle" "kagglehub" "pandas==2.2.2"

import torch
import pandas as pd
import kagglehub

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("pandas version:", pd.__version__)

print("\nTrying target competition download...")
try:
    target_path = kagglehub.competition_download("deep-learning-for-computer-vision-and-nlp-2026-04")
    print("Target competition path:", target_path)
except Exception as e:
    print("Target competition download failed:", repr(e))



CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
pandas version: 2.2.2

Trying target competition download...
Target competition path: /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04


In [17]:
# Stage 2: Inspect target competition files
import os
from collections import defaultdict

if "target_path" not in globals() or not target_path:
    raise RuntimeError("target_path is missing. Run Stage 1 first.")

base_dir = target_path

print("Base dir:", base_dir)

all_files = []
for root, dirs, files in os.walk(base_dir):
    for name in files:
        rel_path = os.path.relpath(os.path.join(root, name), base_dir)
        all_files.append(rel_path)

print("\nTotal files:", len(all_files))

print("\nTop-level entries:")
for name in sorted(os.listdir(base_dir)):
    print(name)

csv_files = sorted([p for p in all_files if p.lower().endswith(".csv")])
zip_files = sorted([p for p in all_files if p.lower().endswith(".zip")])
image_dirs = sorted(
    {
        os.path.dirname(p).split(os.sep)[0]
        for p in all_files
        if p.lower().endswith((".jpg", ".jpeg", ".png"))
    }
)

print("\nCSV files:")
for p in csv_files:
    print(p)

print("\nZIP files:")
for p in zip_files:
    print(p)

print("\nTop image-related dirs:")
for p in image_dirs[:30]:
    print(p)

required_names = {
    "train.csv",
    "test.csv",
    "breed_labels.csv",
    "color_labels.csv",
    "state_labels.csv",
    "sample_submission.csv",
}

resolved = {}
for file_name in required_names:
    matches = [p for p in all_files if os.path.basename(p) == file_name]
    resolved[file_name] = matches

print("\nResolved required files:")
for k, v in resolved.items():
    print(k, "->", v)

missing = [k for k, v in resolved.items() if not v]
print("\nMissing required files:", missing)

Base dir: /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04

Total files: 37923

Top-level entries:
images
sample_submission.csv
test.csv
train.csv

CSV files:
sample_submission.csv
test.csv
train.csv

ZIP files:

Top image-related dirs:
images

Resolved required files:
sample_submission.csv -> ['sample_submission.csv']
test.csv -> ['test.csv']
breed_labels.csv -> []
color_labels.csv -> []
state_labels.csv -> []
train.csv -> ['train.csv']

Missing required files: ['breed_labels.csv', 'color_labels.csv', 'state_labels.csv']


In [19]:
# Stage 3: Dataset analysis
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 140)

train_path = os.path.join(base_dir, "train.csv")
test_path = os.path.join(base_dir, "test.csv")
sample_submission_path = os.path.join(base_dir, "sample_submission.csv")
images_dir = os.path.join(base_dir, "images")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

def build_profile(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        non_null = df[col].dropna()
        sample_values = [str(x) for x in non_null.unique()[:6]]
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "missing": int(df[col].isna().sum()),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "nunique": int(df[col].nunique(dropna=True)),
            "sample_values": sample_values,
        })
    return pd.DataFrame(rows)

train_profile = build_profile(train)
test_profile = build_profile(test)

train_only_cols = sorted(set(train.columns) - set(test.columns))
test_only_cols = sorted(set(test.columns) - set(train.columns))

candidate_target_cols = [c for c in sample_submission.columns if c in train.columns and c not in test.columns]
if not candidate_target_cols:
    candidate_target_cols = train_only_cols[:]

id_candidates = [c for c in train.columns if c.lower() in {"id", "petid", "imageid", "itemid"}]
id_col = id_candidates[0] if id_candidates else None

image_files = []
if os.path.exists(images_dir):
    for name in os.listdir(images_dir):
        if name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_files.append(name)

image_df = pd.DataFrame({"image_file": image_files}) if image_files else pd.DataFrame(columns=["image_file"])

if not image_df.empty:
    image_df["stem"] = image_df["image_file"].str.replace(r"\.[^.]+$", "", regex=True)
    image_df["prefix_before_dash"] = image_df["stem"].str.split("-").str[0]
    image_df["prefix_before_underscore"] = image_df["stem"].str.split("_").str[0]
else:
    image_df["stem"] = []
    image_df["prefix_before_dash"] = []
    image_df["prefix_before_underscore"] = []

print("TRAIN SHAPE:", train.shape)
print("TEST SHAPE:", test.shape)
print("SAMPLE SUBMISSION SHAPE:", sample_submission.shape)

print("\nTRAIN COLUMNS")
print(train.columns.tolist())

print("\nTEST COLUMNS")
print(test.columns.tolist())

print("\nSAMPLE SUBMISSION COLUMNS")
print(sample_submission.columns.tolist())

print("\nTRAIN ONLY COLUMNS:", train_only_cols)
print("TEST ONLY COLUMNS:", test_only_cols)
print("CANDIDATE TARGET COLUMNS:", candidate_target_cols)
print("ID COLUMN CANDIDATE:", id_col)

print("\nTRAIN PROFILE")
print(train_profile.to_string(index=False))

print("\nTEST PROFILE")
print(test_profile.to_string(index=False))

for target_col in candidate_target_cols:
    print(f"\nTARGET DISTRIBUTION: {target_col}")
    print(train[target_col].value_counts(dropna=False).sort_index().to_string())

    if pd.api.types.is_numeric_dtype(train[target_col]):
        print(f"\nTARGET DISTRIBUTION %: {target_col}")
        print((train[target_col].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2).to_string())

numeric_cols = train.select_dtypes(include=["number"]).columns.tolist()
print("\nNUMERIC SUMMARY")
print(train[numeric_cols].describe().round(3).transpose().to_string())

text_like_cols = [c for c in train.columns if train[c].dtype == "object"]
print("\nTEXT-LIKE COLUMNS:", text_like_cols)

for col in text_like_cols[:10]:
    lengths = train[col].fillna("").astype(str).str.len()
    print(f"\nTEXT LENGTH SUMMARY: {col}")
    print(lengths.describe().round(2).to_string())

print("\nIMAGE FILE COUNT:", len(image_files))
print("IMAGE SAMPLE:", image_files[:20])

if id_col and not image_df.empty:
    train_ids = set(train[id_col].astype(str).unique())
    test_ids = set(test[id_col].astype(str).unique())

    by_dash_train = image_df["prefix_before_dash"].isin(train_ids).sum()
    by_dash_test = image_df["prefix_before_dash"].isin(test_ids).sum()
    by_us_train = image_df["prefix_before_underscore"].isin(train_ids).sum()
    by_us_test = image_df["prefix_before_underscore"].isin(test_ids).sum()

    print("\nIMAGE LINK CHECK")
    print("match by prefix_before_dash -> train:", int(by_dash_train))
    print("match by prefix_before_dash -> test :", int(by_dash_test))
    print("match by prefix_before_underscore -> train:", int(by_us_train))
    print("match by prefix_before_underscore -> test :", int(by_us_test))

    train_image_counts_dash = (
        image_df[image_df["prefix_before_dash"].isin(train_ids)]
        .groupby("prefix_before_dash")
        .size()
    )
    test_image_counts_dash = (
        image_df[image_df["prefix_before_dash"].isin(test_ids)]
        .groupby("prefix_before_dash")
        .size()
    )

    print("\nTRAIN IMAGE COUNT SUMMARY BY DASH PREFIX")
    if len(train_image_counts_dash) > 0:
        print(train_image_counts_dash.describe().round(2).to_string())
    else:
        print("No matches")

    print("\nTEST IMAGE COUNT SUMMARY BY DASH PREFIX")
    if len(test_image_counts_dash) > 0:
        print(test_image_counts_dash.describe().round(2).to_string())
    else:
        print("No matches")

print("\nSAMPLE SUBMISSION HEAD")
print(sample_submission.head(10).to_string(index=False))

print("\nTRAIN HEAD")
print(train.head(5).to_string(index=False))

print("\nTEST HEAD")
print(test.head(5).to_string(index=False))

TRAIN SHAPE: (6431, 3)
TEST SHAPE: (1891, 2)
SAMPLE SUBMISSION SHAPE: (4, 2)

TRAIN COLUMNS
['PetID', 'Description', 'AdoptionSpeed']

TEST COLUMNS
['PetID', 'Description']

SAMPLE SUBMISSION COLUMNS
['PetID', 'AdoptionSpeed']

TRAIN ONLY COLUMNS: ['AdoptionSpeed']
TEST ONLY COLUMNS: []
CANDIDATE TARGET COLUMNS: ['AdoptionSpeed']
ID COLUMN CANDIDATE: PetID

TRAIN PROFILE
       column  dtype  missing  missing_pct  nunique                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [20]:
# Stage 4: Recursive image analysis
import os
import glob
import pandas as pd

if "train" not in globals():
    train = pd.read_csv(os.path.join(base_dir, "train.csv"))
if "test" not in globals():
    test = pd.read_csv(os.path.join(base_dir, "test.csv"))

image_paths = [
    p for p in glob.glob(os.path.join(base_dir, "images", "**", "*"), recursive=True)
    if os.path.isfile(p) and p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

image_df = pd.DataFrame({"image_path": image_paths})
image_df["rel_path"] = image_df["image_path"].apply(lambda p: os.path.relpath(p, base_dir))
image_df["file_name"] = image_df["image_path"].apply(os.path.basename)
image_df["stem"] = image_df["file_name"].str.replace(r"\.[^.]+$", "", regex=True)
image_df["petid_dash"] = image_df["stem"].str.split("-").str[0]
image_df["petid_underscore"] = image_df["stem"].str.split("_").str[0]

train_ids = set(train["PetID"].astype(str))
test_ids = set(test["PetID"].astype(str))

dash_train_matches = int(image_df["petid_dash"].isin(train_ids).sum())
dash_test_matches = int(image_df["petid_dash"].isin(test_ids).sum())
underscore_train_matches = int(image_df["petid_underscore"].isin(train_ids).sum())
underscore_test_matches = int(image_df["petid_underscore"].isin(test_ids).sum())

best_image_key = "petid_dash"
if underscore_train_matches + underscore_test_matches > dash_train_matches + dash_test_matches:
    best_image_key = "petid_underscore"

image_df["PetID"] = image_df[best_image_key]

train_image_counts = image_df[image_df["PetID"].isin(train_ids)].groupby("PetID").size()
test_image_counts = image_df[image_df["PetID"].isin(test_ids)].groupby("PetID").size()

print("TOTAL IMAGE FILES:", len(image_df))
print("DASH MATCHES -> train:", dash_train_matches, "test:", dash_test_matches)
print("UNDERSCORE MATCHES -> train:", underscore_train_matches, "test:", underscore_test_matches)
print("BEST IMAGE KEY:", best_image_key)

print("\nTRAIN PETS WITH IMAGES:", int(train_image_counts.shape[0]))
print("TEST PETS WITH IMAGES:", int(test_image_counts.shape[0]))

print("\nTRAIN IMAGE COUNT SUMMARY")
if len(train_image_counts) > 0:
    print(train_image_counts.describe().round(2).to_string())
else:
    print("No matches")

print("\nTEST IMAGE COUNT SUMMARY")
if len(test_image_counts) > 0:
    print(test_image_counts.describe().round(2).to_string())
else:
    print("No matches")

print("\nIMAGE SAMPLE")
print(image_df[["rel_path", "file_name", "PetID"]].head(20).to_string(index=False))

TOTAL IMAGE FILES: 37920
DASH MATCHES -> train: 28472 test: 9359
UNDERSCORE MATCHES -> train: 0 test: 0
BEST IMAGE KEY: petid_dash

TRAIN PETS WITH IMAGES: 6431
TEST PETS WITH IMAGES: 1887

TRAIN IMAGE COUNT SUMMARY
count    6431.00
mean        4.43
std         3.88
min         1.00
25%         2.00
50%         4.00
75%         5.00
max        30.00

TEST IMAGE COUNT SUMMARY
count    1887.00
mean        4.96
std         4.11
min         1.00
25%         2.00
50%         4.00
75%         6.00
max        30.00

IMAGE SAMPLE
                            rel_path        file_name     PetID
 images/images/train/e7ab271b1-8.jpg  e7ab271b1-8.jpg e7ab271b1
 images/images/train/50e061767-9.jpg  50e061767-9.jpg 50e061767
 images/images/train/a3449e0d0-2.jpg  a3449e0d0-2.jpg a3449e0d0
 images/images/train/91c2159dd-2.jpg  91c2159dd-2.jpg 91c2159dd
 images/images/train/711bebb99-1.jpg  711bebb99-1.jpg 711bebb99
 images/images/train/818d4a56e-4.jpg  818d4a56e-4.jpg 818d4a56e
images/images/train/b238

In [21]:
# Stage 5: Install modeling packages
!pip -q install -U "transformers>=4.40,<5" "datasets>=2.18,<4" "accelerate>=0.28" "scikit-learn>=1.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 162.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 138.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 73.9 MB/s eta 0:00:00


In [22]:
# Stage 6: Prepare text datasets
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

base_dir = target_path

train = pd.read_csv(os.path.join(base_dir, "train.csv"))
test = pd.read_csv(os.path.join(base_dir, "test.csv"))
sample_submission = pd.read_csv(os.path.join(base_dir, "sample_submission.csv"))

train["Description"] = train["Description"].fillna("")
test["Description"] = test["Description"].fillna("")

label_values = sorted(train["AdoptionSpeed"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_values)}
id2label = {idx: label for label, idx in label2id.items()}

train["label"] = train["AdoptionSpeed"].map(label2id)

model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

token_lengths = [len(ids) for ids in tokenizer(train["Description"].tolist(), truncation=False, padding=False)["input_ids"]]
max_length = int(np.clip(np.percentile(token_lengths, 95), 64, 256))

train_df, valid_df = train_test_split(
    train[["PetID", "Description", "label", "AdoptionSpeed"]],
    test_size=0.15,
    random_state=42,
    stratify=train["label"]
)

train_ds = Dataset.from_pandas(train_df[["Description", "label"]].reset_index(drop=True))
valid_ds = Dataset.from_pandas(valid_df[["Description", "label"]].reset_index(drop=True))
test_pet_ids = test["PetID"].copy()
test_ds = Dataset.from_pandas(test[["Description"]].reset_index(drop=True))

def tokenize_batch(batch):
    return tokenizer(batch["Description"], truncation=True, max_length=max_length)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["Description"])
valid_ds = valid_ds.map(tokenize_batch, batched=True, remove_columns=["Description"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["Description"])

print("MODEL:", model_name)
print("MAX LENGTH:", max_length)
print("LABELS:", label_values)
print("TRAIN SIZE:", len(train_ds))
print("VALID SIZE:", len(valid_ds))
print("TEST SIZE:", len(test_ds))

print("\nTRAIN CLASS DISTRIBUTION")
print(train_df["AdoptionSpeed"].value_counts(normalize=True).sort_index().round(4).to_string())

print("\nVALID CLASS DISTRIBUTION")
print(valid_df["AdoptionSpeed"].value_counts(normalize=True).sort_index().round(4).to_string())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (1115 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/5466 [00:00<?, ? examples/s]

Map:   0%|          | 0/965 [00:00<?, ? examples/s]

Map:   0%|          | 0/1891 [00:00<?, ? examples/s]

MODEL: distilroberta-base
MAX LENGTH: 256
LABELS: [1, 2, 3, 4]
TRAIN SIZE: 5466
VALID SIZE: 965
TEST SIZE: 1891

TRAIN CLASS DISTRIBUTION
AdoptionSpeed
1    0.1861
2    0.2757
3    0.2065
4    0.3317

VALID CLASS DISTRIBUTION
AdoptionSpeed
1    0.1865
2    0.2756
3    0.2062
4    0.3316


In [26]:
# Stage 7: Train text model
import inspect
import numpy as np
import torch
import transformers
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback

print("transformers version:", transformers.__version__)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_values),
    id2label={i: str(v) for i, v in id2label.items()},
    label2id={str(v): i for v, i in label2id.items()}
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "qwk": cohen_kappa_score(labels, preds, weights="quadratic"),
    }

bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled

training_kwargs = {
    "output_dir": "/content/petfinder_text_model",
    "seed": 42,
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "logging_steps": 50,
    "load_best_model_at_end": True,
    "metric_for_best_model": "qwk",
    "greater_is_better": True,
    "save_total_limit": 1,
    "report_to": [],
    "bf16": bf16_enabled,
    "fp16": fp16_enabled,
    "dataloader_num_workers": 2,
}

ta_params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in ta_params:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": valid_ds,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)],
}

trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()

eval_metrics = trainer.evaluate()
print(eval_metrics)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


transformers version: 4.57.6


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Qwk
1,1.286700,1.275251,0.421762,0.335683,0.250491
2,1.219700,1.279715,0.423834,0.331451,0.194930
3,1.144700,1.294301,0.432124,0.397344,0.280002
4,1.047600,1.326003,0.417617,0.384730,0.302750


{'eval_loss': 1.326003074645996, 'eval_accuracy': 0.41761658031088084, 'eval_macro_f1': 0.3847297669684645, 'eval_qwk': 0.30275034031396064, 'eval_runtime': 0.2458, 'eval_samples_per_second': 3926.131, 'eval_steps_per_second': 126.124, 'epoch': 4.0}


In [27]:
# Stage 8: Inference and submission file
import numpy as np
import pandas as pd

test_logits = trainer.predict(test_ds).predictions
test_pred_ids = np.argmax(test_logits, axis=-1)
test_preds = [int(id2label[int(idx)]) for idx in test_pred_ids]

submission = pd.DataFrame({
    "PetID": test_pet_ids.values,
    "AdoptionSpeed": test_preds
})

submission_path = "/content/submission_text_distilroberta.csv"
submission.to_csv(submission_path, index=False)

print("SUBMISSION SHAPE:", submission.shape)
print("\nPREDICTION DISTRIBUTION")
print(submission["AdoptionSpeed"].value_counts().sort_index().to_string())

print("\nSUBMISSION HEAD")
print(submission.head(10).to_string(index=False))

print("\nSAVED TO:", submission_path)

SUBMISSION SHAPE: (1891, 2)

PREDICTION DISTRIBUTION
AdoptionSpeed
1    293
2    600
3    260
4    738

SUBMISSION HEAD
    PetID  AdoptionSpeed
6697a7f62              2
23b64fe21              4
41e824cbe              3
6c3d7237b              4
97b0b5d92              4
5eb355cea              3
bf2b57c9d              2
4f6029457              2
fe09dbb56              1
9b3e95995              4

SAVED TO: /content/submission_text_distilroberta.csv


In [28]:
# Stage 9: Submit to Kaggle
!kaggle competitions submit -c deep-learning-for-computer-vision-and-nlp-2026-04 -f /content/submission_text_distilroberta.csv -m "distilroberta text baseline"

100% 22.2k/22.2k [00:00<00:00, 51.1kB/s]
Successfully submitted to Deep Learning for Computer Vision and NLP 2026-04

In [29]:
# Stage 10: Prepare image datasets
import os
import glob
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import EfficientNet_B0_Weights

if "base_dir" not in globals():
    base_dir = target_path

if "train" not in globals():
    train = pd.read_csv(os.path.join(base_dir, "train.csv"))
if "test" not in globals():
    test = pd.read_csv(os.path.join(base_dir, "test.csv"))

if "label" not in train.columns:
    train["label"] = train["AdoptionSpeed"].map(label2id)

if "image_df" not in globals() or image_df.empty:
    image_paths = [
        p for p in glob.glob(os.path.join(base_dir, "images", "**", "*"), recursive=True)
        if os.path.isfile(p) and p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]
    image_df = pd.DataFrame({"image_path": image_paths})
    image_df["rel_path"] = image_df["image_path"].apply(lambda p: os.path.relpath(p, base_dir))
    image_df["file_name"] = image_df["image_path"].apply(os.path.basename)
    image_df["stem"] = image_df["file_name"].str.replace(r"\.[^.]+$", "", regex=True)
    image_df["PetID"] = image_df["stem"].str.split("-").str[0]

train_pet_ids = set(train_df["PetID"].astype(str))
valid_pet_ids = set(valid_df["PetID"].astype(str))
test_pet_ids = set(test["PetID"].astype(str))

pet_label_map = train[["PetID", "label"]].drop_duplicates().set_index("PetID")["label"].to_dict()

image_train_df = image_df[image_df["PetID"].isin(train_pet_ids)][["PetID", "image_path"]].copy()
image_valid_df = image_df[image_df["PetID"].isin(valid_pet_ids)][["PetID", "image_path"]].copy()
image_test_df = image_df[image_df["PetID"].isin(test_pet_ids)][["PetID", "image_path"]].copy()

image_train_df["label"] = image_train_df["PetID"].map(pet_label_map)
image_valid_df["label"] = image_valid_df["PetID"].map(pet_label_map)

weights = EfficientNet_B0_Weights.DEFAULT
image_transform = weights.transforms()

class PetImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, has_label: bool):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        item = {
            "pixel_values": image,
            "PetID": row["PetID"],
        }
        if self.has_label:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

image_train_ds = PetImageDataset(image_train_df, image_transform, has_label=True)
image_valid_ds = PetImageDataset(image_valid_df, image_transform, has_label=True)
image_test_ds = PetImageDataset(image_test_df, image_transform, has_label=False)

train_loader = DataLoader(image_train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(image_valid_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(image_test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("IMAGE TRAIN ROWS:", len(image_train_df))
print("IMAGE VALID ROWS:", len(image_valid_df))
print("IMAGE TEST ROWS:", len(image_test_df))
print("TRAIN PETS:", image_train_df['PetID'].nunique())
print("VALID PETS:", image_valid_df['PetID'].nunique())
print("TEST PETS:", image_test_df['PetID'].nunique())
print(image_train_df.head(10).to_string(index=False))

IMAGE TRAIN ROWS: 24324
IMAGE VALID ROWS: 4148
IMAGE TEST ROWS: 9359
TRAIN PETS: 5466
VALID PETS: 965
TEST PETS: 1887
    PetID                                                                                                                 image_path  label
e7ab271b1  /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04/images/images/train/e7ab271b1-8.jpg      2
50e061767  /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04/images/images/train/50e061767-9.jpg      1
91c2159dd  /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04/images/images/train/91c2159dd-2.jpg      3
711bebb99  /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04/images/images/train/711bebb99-1.jpg      3
818d4a56e  /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-04/images/images/train/818d4a56e-4.jpg      0
b2385d700 /root/.cache/kagglehub/competiti

In [31]:
# Stage 11: Train image model
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled
amp_enabled = bf16_enabled or fp16_enabled
amp_dtype = torch.bfloat16 if bf16_enabled else torch.float16

image_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
in_features = image_model.classifier[1].in_features
image_model.classifier[1] = nn.Linear(in_features, len(label_values))
image_model = image_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(image_model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=fp16_enabled)

prob_cols = [f"img_prob_{i}" for i in range(len(label_values))]

def collect_pet_probs(model, loader, has_label: bool):
    model.eval()
    petids = []
    probs_list = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            x = batch["pixel_values"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                logits = model(x)

            probs = torch.softmax(logits.float(), dim=-1).detach().cpu().numpy()
            probs_list.append(probs)
            petids.extend(batch["PetID"])

            if has_label:
                labels.extend(batch["label"].cpu().numpy().tolist())

    df = pd.DataFrame(np.vstack(probs_list), columns=prob_cols)
    df.insert(0, "PetID", petids)

    if has_label:
        df["label"] = labels
        agg_probs = df.groupby("PetID")[prob_cols].mean().reset_index()
        agg_labels = df.groupby("PetID")["label"].first().reset_index()
        return agg_probs.merge(agg_labels, on="PetID", how="left")

    return df.groupby("PetID")[prob_cols].mean().reset_index()

best_qwk = -1.0
best_state = None
history = []

for epoch in range(1, 4):
    image_model.train()
    running_loss = 0.0
    seen = 0

    for batch in train_loader:
        x = batch["pixel_values"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
            logits = image_model(x)
            loss = criterion(logits, y)

        if fp16_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        seen += batch_size

    image_valid_probs = collect_pet_probs(image_model, valid_loader, has_label=True)
    valid_true = image_valid_probs["label"].to_numpy()
    valid_pred = image_valid_probs[prob_cols].to_numpy().argmax(axis=1)

    valid_acc = accuracy_score(valid_true, valid_pred)
    valid_f1 = f1_score(valid_true, valid_pred, average="macro")
    valid_qwk = cohen_kappa_score(valid_true, valid_pred, weights="quadratic")
    train_loss = running_loss / max(seen, 1)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_accuracy": valid_acc,
        "valid_macro_f1": valid_f1,
        "valid_qwk": valid_qwk,
    })

    print({
        "epoch": epoch,
        "train_loss": round(train_loss, 5),
        "valid_accuracy": round(valid_acc, 5),
        "valid_macro_f1": round(valid_f1, 5),
        "valid_qwk": round(valid_qwk, 5),
    })

    if valid_qwk > best_qwk:
        best_qwk = valid_qwk
        best_state = copy.deepcopy(image_model.state_dict())

image_model.load_state_dict(best_state)

image_valid_probs = collect_pet_probs(image_model, valid_loader, has_label=True)
image_test_probs = collect_pet_probs(image_model, test_loader, has_label=False)

print("\nBEST IMAGE QWK:", best_qwk)
print("\nIMAGE VALID PROBS HEAD")
print(image_valid_probs.head(10).to_string(index=False))

print("\nIMAGE TEST PROBS HEAD")
print(image_test_probs.head(10).to_string(index=False))

{'epoch': 1, 'train_loss': 1.22136, 'valid_accuracy': 0.51503, 'valid_macro_f1': 0.47788, 'valid_qwk': 0.47497}
{'epoch': 2, 'train_loss': 1.00661, 'valid_accuracy': 0.50052, 'valid_macro_f1': 0.46796, 'valid_qwk': 0.46963}
{'epoch': 3, 'train_loss': 0.75005, 'valid_accuracy': 0.50052, 'valid_macro_f1': 0.47518, 'valid_qwk': 0.46364}

BEST IMAGE QWK: 0.47496851283984776

IMAGE VALID PROBS HEAD
    PetID  img_prob_0  img_prob_1  img_prob_2  img_prob_3  label
009239f92    0.206750    0.454757    0.276550    0.061943      2
00c71a320    0.018613    0.100149    0.133159    0.748079      3
00f63c6fc    0.100503    0.137371    0.273196    0.488930      3
0111f92ff    0.278516    0.259983    0.087554    0.373947      0
01573c686    0.021654    0.090328    0.090621    0.797396      3
0164a9cfb    0.060487    0.188928    0.317125    0.433461      1
01802debf    0.188353    0.236501    0.315899    0.259247      2
01928bd13    0.290822    0.229506    0.279965    0.199707      0
01b167b92    0.141

In [32]:
# Stage 12: Collect text probabilities
import numpy as np
import pandas as pd

text_valid_logits = trainer.predict(valid_ds).predictions
text_test_logits = trainer.predict(test_ds).predictions

text_valid_probs_np = torch.softmax(torch.tensor(text_valid_logits), dim=-1).cpu().numpy()
text_test_probs_np = torch.softmax(torch.tensor(text_test_logits), dim=-1).cpu().numpy()

text_prob_cols = [f"text_prob_{i}" for i in range(len(label_values))]

text_valid_probs = pd.DataFrame(text_valid_probs_np, columns=text_prob_cols)
text_valid_probs.insert(0, "PetID", valid_df["PetID"].astype(str).values)
text_valid_probs["label"] = valid_df["label"].values

text_test_probs = pd.DataFrame(text_test_probs_np, columns=text_prob_cols)
text_test_probs.insert(0, "PetID", test["PetID"].astype(str).values)

print("TEXT VALID PROBS SHAPE:", text_valid_probs.shape)
print("TEXT TEST PROBS SHAPE:", text_test_probs.shape)

print("\nTEXT VALID PROBS HEAD")
print(text_valid_probs.head(10).to_string(index=False))

print("\nTEXT TEST PROBS HEAD")
print(text_test_probs.head(10).to_string(index=False))

TEXT VALID PROBS SHAPE: (965, 6)
TEXT TEST PROBS SHAPE: (1891, 5)

TEXT VALID PROBS HEAD
    PetID  text_prob_0  text_prob_1  text_prob_2  text_prob_3  label
8a4e70a21     0.223231     0.483787     0.269268     0.023713      1
5ba0605b9     0.076013     0.062161     0.052550     0.809276      3
59b6102ba     0.455602     0.434739     0.103360     0.006299      1
975091325     0.099722     0.131467     0.072532     0.696278      3
3cdbf07be     0.604278     0.320930     0.067303     0.007489      0
bd7b805ba     0.033866     0.056604     0.046289     0.863240      3
648897870     0.381504     0.238273     0.115447     0.264777      0
65f311c09     0.302956     0.375565     0.171359     0.150120      3
b70f7c3d6     0.069987     0.078996     0.130235     0.720782      1
45fd43894     0.533851     0.281320     0.088176     0.096653      3

TEXT TEST PROBS HEAD
    PetID  text_prob_0  text_prob_1  text_prob_2  text_prob_3
6697a7f62     0.141227     0.520638     0.330937     0.007198
23b64f

In [33]:
# Stage 13: Fuse text and image predictions
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

text_prob_cols = [f"text_prob_{i}" for i in range(len(label_values))]
img_prob_cols = [f"img_prob_{i}" for i in range(len(label_values))]

valid_fusion = text_valid_probs.merge(
    image_valid_probs[["PetID"] + img_prob_cols],
    on="PetID",
    how="left"
)

test_fusion = text_test_probs.merge(
    image_test_probs[["PetID"] + img_prob_cols],
    on="PetID",
    how="left"
)

valid_text_np = valid_fusion[text_prob_cols].to_numpy()
valid_img_np = valid_fusion[img_prob_cols].to_numpy()
valid_has_image = ~np.isnan(valid_img_np).all(axis=1)
valid_img_np = np.nan_to_num(valid_img_np, nan=0.0)
valid_true = valid_fusion["label"].to_numpy()

best_result = None

for text_weight in np.linspace(0.0, 1.0, 21):
    fused = valid_text_np.copy()
    fused[valid_has_image] = text_weight * valid_text_np[valid_has_image] + (1.0 - text_weight) * valid_img_np[valid_has_image]

    pred_ids = fused.argmax(axis=1)

    acc = accuracy_score(valid_true, pred_ids)
    macro_f1 = f1_score(valid_true, pred_ids, average="macro")
    qwk = cohen_kappa_score(valid_true, pred_ids, weights="quadratic")

    result = {
        "text_weight": float(text_weight),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "qwk": float(qwk),
    }

    if best_result is None or result["qwk"] > best_result["qwk"]:
        best_result = result

print("BEST FUSION:", best_result)

test_text_np = test_fusion[text_prob_cols].to_numpy()
test_img_np = test_fusion[img_prob_cols].to_numpy()
test_has_image = ~np.isnan(test_img_np).all(axis=1)
test_img_np = np.nan_to_num(test_img_np, nan=0.0)

best_text_weight = best_result["text_weight"]

test_fused = test_text_np.copy()
test_fused[test_has_image] = best_text_weight * test_text_np[test_has_image] + (1.0 - best_text_weight) * test_img_np[test_has_image]

test_pred_ids = test_fused.argmax(axis=1)
test_preds = [int(id2label[int(idx)]) for idx in test_pred_ids]

submission_fusion = pd.DataFrame({
    "PetID": test_fusion["PetID"].values,
    "AdoptionSpeed": test_preds
})

submission_fusion_path = "/content/submission_multimodal_fusion.csv"
submission_fusion.to_csv(submission_fusion_path, index=False)

print("\nFUSION PREDICTION DISTRIBUTION")
print(submission_fusion["AdoptionSpeed"].value_counts().sort_index().to_string())

print("\nFUSION SUBMISSION HEAD")
print(submission_fusion.head(10).to_string(index=False))

print("\nSAVED TO:", submission_fusion_path)

BEST FUSION: {'text_weight': 0.25, 'accuracy': 0.5243523316062176, 'macro_f1': 0.4793924308369346, 'qwk': 0.5041413064600503}

FUSION PREDICTION DISTRIBUTION
AdoptionSpeed
1    284
2    529
3    198
4    880

FUSION SUBMISSION HEAD
    PetID  AdoptionSpeed
6697a7f62              4
23b64fe21              2
41e824cbe              4
6c3d7237b              4
97b0b5d92              4
5eb355cea              4
bf2b57c9d              2
4f6029457              1
fe09dbb56              2
9b3e95995              4

SAVED TO: /content/submission_multimodal_fusion.csv


In [34]:
# Stage 14: Submit multimodal result
!kaggle competitions submit -c deep-learning-for-computer-vision-and-nlp-2026-04 -f /content/submission_multimodal_fusion.csv -m "text image late fusion baseline"

100% 22.2k/22.2k [00:00<00:00, 54.8kB/s]
Successfully submitted to Deep Learning for Computer Vision and NLP 2026-04

In [37]:
# Stage 15: Image TTA inference and fusion
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import EfficientNet_B0_Weights
from torchvision.transforms import functional as TF
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled
amp_enabled = bf16_enabled or fp16_enabled
amp_dtype = torch.bfloat16 if bf16_enabled else torch.float16

weights = EfficientNet_B0_Weights.DEFAULT
base_transform = weights.transforms()

tta_modes = ["base", "hflip"]

class PetImageTTADataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, tta_mode: str, has_label: bool):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.tta_mode = tta_mode
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        if self.tta_mode == "hflip":
            image = TF.hflip(image)

        image = self.transform(image)

        item = {
            "pixel_values": image,
            "PetID": row["PetID"],
        }
        if self.has_label:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

prob_cols = [f"img_prob_{i}" for i in range(len(label_values))]
text_prob_cols = [f"text_prob_{i}" for i in range(len(label_values))]
img_prob_cols = [f"img_prob_{i}" for i in range(len(label_values))]

def collect_pet_probs_tta(model, df: pd.DataFrame, has_label: bool):
    tta_outputs = []
    label_map = None

    if has_label:
        label_map = df[["PetID", "label"]].drop_duplicates().copy()

    for tta_mode in tta_modes:
        ds = PetImageTTADataset(df, transform=base_transform, tta_mode=tta_mode, has_label=has_label)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

        model.eval()
        petids = []
        probs_list = []

        with torch.no_grad():
            for batch in loader:
                x = batch["pixel_values"].to(device, non_blocking=True)

                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                    logits = model(x)

                probs = torch.softmax(logits.float(), dim=-1).detach().cpu().numpy()
                probs_list.append(probs)
                petids.extend(batch["PetID"])

        tmp = pd.DataFrame(np.vstack(probs_list), columns=prob_cols)
        tmp.insert(0, "PetID", petids)
        tmp = tmp.groupby("PetID", as_index=False)[prob_cols].mean()
        tta_outputs.append(tmp)

    merged = tta_outputs[0][["PetID"]].copy()
    for col in prob_cols:
        merged[col] = 0.0

    for tmp in tta_outputs:
        merged = merged.merge(tmp, on="PetID", how="left", suffixes=("", "_tmp"))
        for col in prob_cols:
            tmp_col = f"{col}_tmp" if f"{col}_tmp" in merged.columns else col
            merged[col] = merged[col] + merged[tmp_col]
            if tmp_col != col:
                merged.drop(columns=[tmp_col], inplace=True)

    for col in prob_cols:
        merged[col] = merged[col] / len(tta_outputs)

    if has_label:
        merged = merged.merge(label_map, on="PetID", how="left")

    return merged

image_valid_probs_tta = collect_pet_probs_tta(image_model, image_valid_df, has_label=True)
image_test_probs_tta = collect_pet_probs_tta(image_model, image_test_df, has_label=False)

valid_true = image_valid_probs_tta["label"].to_numpy()
valid_pred = image_valid_probs_tta[prob_cols].to_numpy().argmax(axis=1)

image_tta_acc = accuracy_score(valid_true, valid_pred)
image_tta_f1 = f1_score(valid_true, valid_pred, average="macro")
image_tta_qwk = cohen_kappa_score(valid_true, valid_pred, weights="quadratic")

print("IMAGE TTA METRICS")
print({
    "accuracy": round(image_tta_acc, 5),
    "macro_f1": round(image_tta_f1, 5),
    "qwk": round(image_tta_qwk, 5),
})

valid_fusion_tta = text_valid_probs.merge(
    image_valid_probs_tta[["PetID"] + img_prob_cols],
    on="PetID",
    how="left"
)

test_fusion_tta = text_test_probs.merge(
    image_test_probs_tta[["PetID"] + img_prob_cols],
    on="PetID",
    how="left"
)

valid_text_np = valid_fusion_tta[text_prob_cols].to_numpy()
valid_img_np = valid_fusion_tta[img_prob_cols].to_numpy()
valid_has_image = ~np.isnan(valid_img_np).all(axis=1)
valid_img_np = np.nan_to_num(valid_img_np, nan=0.0)
valid_true = valid_fusion_tta["label"].to_numpy()

best_result_tta = None

for text_weight in np.linspace(0.0, 1.0, 41):
    fused = valid_text_np.copy()
    fused[valid_has_image] = text_weight * valid_text_np[valid_has_image] + (1.0 - text_weight) * valid_img_np[valid_has_image]

    pred_ids = fused.argmax(axis=1)

    acc = accuracy_score(valid_true, pred_ids)
    macro_f1 = f1_score(valid_true, pred_ids, average="macro")
    qwk = cohen_kappa_score(valid_true, pred_ids, weights="quadratic")

    result = {
        "text_weight": float(text_weight),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "qwk": float(qwk),
    }

    if best_result_tta is None or result["qwk"] > best_result_tta["qwk"]:
        best_result_tta = result

print("\nBEST FUSION TTA:", best_result_tta)

test_text_np = test_fusion_tta[text_prob_cols].to_numpy()
test_img_np = test_fusion_tta[img_prob_cols].to_numpy()
test_has_image = ~np.isnan(test_img_np).all(axis=1)
test_img_np = np.nan_to_num(test_img_np, nan=0.0)

best_text_weight_tta = best_result_tta["text_weight"]

test_fused_tta = test_text_np.copy()
test_fused_tta[test_has_image] = best_text_weight_tta * test_text_np[test_has_image] + (1.0 - best_text_weight_tta) * test_img_np[test_has_image]

test_pred_ids_tta = test_fused_tta.argmax(axis=1)
test_preds_tta = [int(id2label[int(idx)]) for idx in test_pred_ids_tta]

submission_tta = pd.DataFrame({
    "PetID": test_fusion_tta["PetID"].values,
    "AdoptionSpeed": test_preds_tta
})

submission_tta_path = "/content/submission_multimodal_fusion_tta.csv"
submission_tta.to_csv(submission_tta_path, index=False)

print("\nTTA PREDICTION DISTRIBUTION")
print(submission_tta["AdoptionSpeed"].value_counts().sort_index().to_string())

print("\nTTA SUBMISSION HEAD")
print(submission_tta.head(10).to_string(index=False))

print("\nSAVED TO:", submission_tta_path)

IMAGE TTA METRICS
{'accuracy': 0.5057, 'macro_f1': 0.46646, 'qwk': 0.45892}

BEST FUSION TTA: {'text_weight': 0.275, 'accuracy': 0.5170984455958549, 'macro_f1': 0.468321452727247, 'qwk': 0.49685690869410304}

TTA PREDICTION DISTRIBUTION
AdoptionSpeed
1    280
2    531
3    198
4    882

TTA SUBMISSION HEAD
    PetID  AdoptionSpeed
6697a7f62              4
23b64fe21              2
41e824cbe              4
6c3d7237b              4
97b0b5d92              4
5eb355cea              4
bf2b57c9d              2
4f6029457              1
fe09dbb56              2
9b3e95995              4

SAVED TO: /content/submission_multimodal_fusion_tta.csv


In [38]:
# Stage 16: Submit TTA multimodal result
!kaggle competitions submit -c deep-learning-for-computer-vision-and-nlp-2026-04 -f /content/submission_multimodal_fusion_tta.csv -m "text image fusion with image tta"

100% 22.2k/22.2k [00:00<00:00, 58.9kB/s]
Successfully submitted to Deep Learning for Computer Vision and NLP 2026-04